# OpenMed Benchmark Evaluation Walkthrough

This notebook demonstrates how to evaluate entity extraction models and rule-based detectors against OpenMed's bundled **synthetic golden benchmarks**.

Evaluation workflow:
1. Load versioned multilingual benchmark fixtures with `openmed.eval.golden.loader.load_golden_fixtures()`.
2. Execute extraction against fixture texts offline.
3. Compute standard information retrieval metrics (Precision, Recall, F1 score, Exact Matches).

> **Synthetic Benchmark Invariant**: Golden fixtures are entirely **synthetic** and designed for deterministic, offline regression testing.

In [1]:
import logging
from openmed.core.safety_sweep import safety_sweep
from openmed.eval.golden.loader import load_golden_fixtures

# Suppress first-run download telemetry for offline execution
logging.getLogger("openmed.core.models").setLevel(logging.ERROR)

fixtures = load_golden_fixtures()
print(f"Loaded {len(fixtures)} synthetic golden evaluation fixtures.")

Loaded 206 synthetic golden evaluation fixtures.


## 1. Inspect Fixture Categories and Locales

We inspect the category distribution and supported languages across the loaded benchmark fixtures.

In [2]:
categories = {}
languages = set()

for fixture in fixtures:
    categories[fixture.category] = categories.get(fixture.category, 0) + 1
    languages.add(fixture.language)

print("=== Golden Benchmark Distribution ===")
print(f"Languages Covered ({len(languages)}): {sorted(languages)}")
print("\nCategories:")
for cat, count in sorted(categories.items()):
    print(f"  - {cat}: {count} fixtures")

print("\n=== Sample Fixtures ===")
for fixture in fixtures[:3]:
    print(f"- ID: {fixture.fixture_id} | Lang: {fixture.language} | Gold Spans: {len(fixture.gold_spans)}")

=== Golden Benchmark Distribution ===
Languages Covered (53): ['af', 'am', 'ar', 'as', 'bg', 'bn', 'cs', 'da', 'de', 'el', 'en', 'es', 'et', 'fi', 'fr', 'gu', 'he', 'hi', 'hr', 'hu', 'id', 'ig', 'it', 'ja', 'kn', 'ko', 'lv', 'ml', 'mr', 'ms', 'nl', 'no', 'or', 'pa', 'pt', 'ro', 'ru', 'sk', 'sr', 'sv', 'sw', 'ta', 'te', 'th', 'tl', 'tr', 'uk', 'ur', 'vi', 'xh', 'yo', 'zh', 'zu']

Categories:
  - checksum_ids: 18 fixtures
  - chunk_boundary: 1 fixtures
  - critical_findings: 1 fixtures
  - date_arithmetic: 1 fixtures
  - financial_ids: 3 fixtures
  - hard_negatives: 8 fixtures
  - india_health_ids: 1 fixtures
  - multilingual: 168 fixtures
  - nested_overlapping: 1 fixtures
  - policy_profile_actions: 4 fixtures

=== Sample Fixtures ===
- ID: golden-checksum-be-rrn | Lang: nl | Gold Spans: 2
- ID: golden-checksum-ch-ahv | Lang: de | Gold Spans: 1
- ID: golden-financial-ids-en | Lang: en | Gold Spans: 2


## 2. Execute Offline Extraction and Compute Metrics

We run the deterministic `safety_sweep` detector across a sample of fixtures and compute standard information-retrieval metrics against the recorded gold spans.

In [3]:
total_gold = 0
total_pred = 0
true_positives = 0
exact_matches = 0

# Evaluate on first 10 fixtures
eval_sample = fixtures[:10]

for fixture in eval_sample:
    pred_spans = safety_sweep(fixture.text, [], lang=fixture.language)
    gold_spans = fixture.gold_spans

    total_gold += len(gold_spans)
    total_pred += len(pred_spans)

    for gold in gold_spans:
        for pred in pred_spans:
            # Overlap evaluation
            if max(gold.start, pred.start) < min(gold.end, pred.end):
                true_positives += 1
                if gold.start == pred.start and gold.end == pred.end:
                    exact_matches += 1
                break

precision = true_positives / total_pred if total_pred > 0 else 0.0
recall = true_positives / total_gold if total_gold > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print("=== Evaluation Report ===")
print(f"Fixtures Evaluated:  {len(eval_sample)}")
print(f"Total Gold Spans:    {total_gold}")
print(f"Total Predictions:   {total_pred}")
print(f"True Positives:      {true_positives}")
print(f"Exact Matches:       {exact_matches}")
print(f"Precision:           {precision:.4f}")
print(f"Recall:              {recall:.4f}")
print(f"F1 Score:            {f1:.4f}")

=== Evaluation Report ===
Fixtures Evaluated:  10
Total Gold Spans:    18
Total Predictions:   12
True Positives:      12
Exact Matches:       10
Precision:           1.0000
Recall:              0.6667
F1 Score:            0.8000
